In [15]:
import pandas as pd
import numpy as np
import sqlite3
import statsmodels.stats.multitest as mt
from scipy import stats


In [16]:
with sqlite3.connect("../data/pancreas_proteomics.db") as conn:
    df = pd.read_sql("SELECT * FROM imputed_MNAR", conn)
    df_ann = pd.read_sql("SELECT [Annotated Matrisome Category], COUNT([Protein ID]) as k_count FROM annotations GROUP BY [Annotated Matrisome Category]", conn)
    df_ann2 = pd.read_sql("SELECT [Annotated Matrisome Division], COUNT([Protein ID]) as div_count FROM annotations GROUP BY [Annotated Matrisome Division]", conn)


#df
df_pivot = df.pivot(index='Protein ID', columns='Sample ID', values='Intensity')
#df_pivot

In [17]:
results = []
for protein in df_pivot.index:
    index = df_pivot.index.get_loc(protein)
    t1d = df_pivot.iloc[index, list(range(9))].dropna()
    ctrl = df_pivot.iloc[index, list(range(9,18))].dropna() 
    #print(ctrl)

    t_stat, p_val = stats.ttest_ind(t1d, ctrl, equal_var=False)
    fc = t1d.mean() - ctrl.mean()

    results.append((protein, p_val, fc))

results_df = pd.DataFrame(results, columns=['Protein ID', 'Log2FC', 'p_value']).dropna()
_, results_df['adj_p_value'], _, _ = mt.multipletests(results_df['p_value'], method='fdr_bh')

    

c:\amish\pancrea_proteomics\t1d_proteomics_pipeline\.venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:601: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


In [18]:
dep = results_df[(results_df['adj_p_value'] < 0.05) & (results_df['Log2FC'].abs() > 0.58)]
dep

,Protein ID,Log2FC,p_value,adj_p_value
9,A1A5D9,0.728687,-0.413012,-3.618598
23,A6NHG4,0.918309,-0.034120,-0.128183
24,A6NHL2,0.768284,-0.044010,-0.169839
26,A6NHR9,0.935956,0.010214,0.034364
30,A6NMZ7,0.969729,-0.024467,-0.089741
...,...,...,...,...
4942,Q9Y6I3,0.855700,-0.037654,-0.142540
4945,Q9Y6K0,0.784409,-0.052377,-0.207623
4946,Q9Y6K9,0.925776,-0.058131,-0.234942
4955,Q9Y6W5,0.953199,-0.057905,-0.233837


In [19]:
N = len(results_df)
n = len(dep)
significant_proteins = dep['Protein ID'].to_list()
tested_proteins = results_df['Protein ID'].to_list()
N, n

(4959, 458)

In [20]:
for index, row in df_ann.iterrows():
    category = row['Annotated Matrisome Category']
    cat_protein = pd.read_sql(f"SELECT [Protein ID] FROM annotations WHERE [Annotated Matrisome Category] = '{category}'", conn)['Protein ID'].to_list()
    cat_protein_tested = set(cat_protein).intersection(set(tested_proteins))
    K = len(cat_protein_tested)
    
    if K == 0:
        continue

    k = len(set(cat_protein).intersection(set(significant_proteins)))
    p_value = stats.hypergeom.sf(k-1, N, K, n)
    print(f"Significance of {category} is {p_value}, {k-1}, {K}, {n}, {N}")
    


Significance of Collagens is 0.08558265763468581, 4, 26, 458, 4959
Significance of ECM Glycoproteins is 0.013678849671058046, 13, 80, 458, 4959
Significance of ECM Regulators is 0.24090849351620727, 8, 74, 458, 4959
Significance of ECM-affiliated Proteins is 0.13300581154702068, 5, 38, 458, 4959
Significance of Non-matrisome is 0.9999454870728303, 416, 4712, 458, 4959
Significance of Proteoglycans is 0.0030073359415679762, 4, 12, 458, 4959
Significance of Secreted Factors is 0.4746410893552236, 1, 17, 458, 4959


In [23]:
for index, row in df_ann2.iterrows():
    category = row['Annotated Matrisome Division']
    cat_protein = pd.read_sql(f"SELECT [Protein ID] FROM annotations WHERE [Annotated Matrisome Division] = '{category}'", conn)['Protein ID'].to_list()
    cat_protein_tested = set(cat_protein).intersection(set(tested_proteins))
    K = len(cat_protein_tested)
    
    if K == 0:
        continue

    k = len(set(cat_protein).intersection(set(significant_proteins)))
    p_value = stats.hypergeom.sf(k-1, N, K, n)
    print(f"Significance of {category} is {p_value}, {k-1}, {K}, {n}, {N}")

    

Significance of Core matrisome is 0.00014767019789242922, 23, 118, 458, 4959
Significance of Matrisome-associated is 0.08350550723862249, 16, 129, 458, 4959
Significance of Non-matrisome is 0.9999454870728303, 416, 4712, 458, 4959
